# 🛡️ SentinelID — Master A100 Training Notebook

Runs all 7 modules back-to-back on **real public datasets** with synthetic fallback.

| Module | Architecture | Dataset |
|--------|-------------|----------|
| M1 Liveness | DepthLivenessModel / ResNet-50 | CelebA-Spoof (625k) |
| M2 Deepfake | EfficientNet-B4 + FFT | Celeb-DF v2 |
| M3 Face Recog | ArcFace / iResNet-100 | MS1M-ArcFace (5.8M) |
| M4 Behavioral | AU-GNN + Gaze | Synthetic landmarks |
| M5 Document | LayoutLM-style | MIDV-500 |
| M6 Fusion | Calibrated MLP | Derived scores |
| M7 Edge | MobileNetV3 distilled | From M1 teacher |

**Estimated time on A100:** ~10–15 h real datasets · ~45 min synthetic fallback

## 0a · Keep-Alive — Run First!

In [ ]:
import threading, time

def _keep_alive():
    while True:
        time.sleep(30)

threading.Thread(target=_keep_alive, daemon=True).start()
print('✓ Keep-alive thread started')

try:
    from google.colab import output
    output.eval_js('''
        setInterval(() => {
            const btn = document.querySelector("colab-toolbar-button#connect");
            if (btn) btn.click();
        }, 60000);
        console.log("Anti-disconnect JS active");
    ''')
except Exception:
    pass
print('✓ Browser anti-disconnect injected')

## 0b · Setup — Run Once

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE_CKPT = pathlib.Path('/content/drive/MyDrive/sentinelid/checkpoints')
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
print('Drive mounted. Checkpoint dir:', DRIVE_CKPT)

In [ ]:
import subprocess, os

REPO = '/content/SentinelID'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/Aprameya05/SentinelID.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--rebase=false'], check=True)

os.chdir(REPO)
print('Repo at:', os.getcwd())

In [ ]:
!pip install -q timm omegaconf wandb rich einops onnx onnxruntime torchmetrics gdown
!pip install -q torch-geometric torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-$(python -c 'import torch; print(torch.__version__.split("+")[0])')+cu121.html
print('✓ Dependencies installed.')

In [ ]:
import wandb
wandb.login()

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1 · Download Real Public Datasets

All freely available — no registration. Each cell checks Drive cache first.
If a download fails the synthetic fallback cell below fills the gap automatically.

In [ ]:
# ── M1: CelebA-Spoof ──────────────────────────────────────────────────────────
import gdown, zipfile, os, shutil
from pathlib import Path

LIVENESS_ROOT = Path('/content/data/liveness')
DRIVE_LIV = Path('/content/drive/MyDrive/sentinelid/data/liveness')

if DRIVE_LIV.exists() and sum(1 for _ in DRIVE_LIV.rglob('*.jpg')) > 1000:
    print('CelebA-Spoof found on Drive — symlinking...')
    LIVENESS_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if not LIVENESS_ROOT.exists():
        os.symlink(str(DRIVE_LIV), str(LIVENESS_ROOT))
    print(f'✓ {sum(1 for _ in LIVENESS_ROOT.rglob("*.jpg"))} images ready')
else:
    LIVENESS_ROOT.mkdir(parents=True, exist_ok=True)
    try:
        print('Downloading CelebA-Spoof subset (~4 GB)...')
        gdown.download_folder(
            url='https://drive.google.com/drive/folders/1OW_1bawO0zLNeMckQes2ojmRAS5NKNHX',
            output=str(LIVENESS_ROOT), quiet=False, resume=True,
        )
        DRIVE_LIV.mkdir(parents=True, exist_ok=True)
        shutil.copytree(str(LIVENESS_ROOT), str(DRIVE_LIV), dirs_exist_ok=True)
        print('✓ CelebA-Spoof downloaded and cached')
    except Exception as e:
        print(f'⚠ Download failed ({e}) — synthetic fallback will run')
        LIVENESS_ROOT = None

print('LIVENESS_ROOT =', LIVENESS_ROOT)

In [ ]:
# ── M2: Celeb-DF v2 frames ────────────────────────────────────────────────────
import gdown, zipfile, os, shutil
from pathlib import Path

DEEPFAKE_ROOT = Path('/content/data/deepfake')
DRIVE_DF = Path('/content/drive/MyDrive/sentinelid/data/deepfake')

if DRIVE_DF.exists() and sum(1 for _ in DRIVE_DF.rglob('*.jpg')) > 1000:
    print('Celeb-DF frames found on Drive — symlinking...')
    DEEPFAKE_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if not DEEPFAKE_ROOT.exists():
        os.symlink(str(DRIVE_DF), str(DEEPFAKE_ROOT))
    print(f'✓ {sum(1 for _ in DEEPFAKE_ROOT.rglob("*.jpg"))} frames ready')
else:
    DEEPFAKE_ROOT.mkdir(parents=True, exist_ok=True)
    try:
        print('Downloading Celeb-DF v2 pre-extracted frames (~8 GB)...')
        gdown.download(
            'https://drive.google.com/uc?id=1iLx76s_bZkNfvmRNMvrF-UuwHOrT0kZX',
            '/content/celebdf_frames.zip', quiet=False
        )
        with zipfile.ZipFile('/content/celebdf_frames.zip') as z:
            z.extractall(str(DEEPFAKE_ROOT))
        DRIVE_DF.mkdir(parents=True, exist_ok=True)
        shutil.copytree(str(DEEPFAKE_ROOT), str(DRIVE_DF), dirs_exist_ok=True)
        print('✓ Celeb-DF v2 ready')
    except Exception as e:
        print(f'⚠ Download failed ({e}) — synthetic fallback will run')
        DEEPFAKE_ROOT = None

print('DEEPFAKE_ROOT =', DEEPFAKE_ROOT)

In [ ]:
# ── M3: MS1M-ArcFace (InsightFace) ────────────────────────────────────────────
import gdown, zipfile, os, shutil
from pathlib import Path

FACE_ROOT = Path('/content/data/faces')
DRIVE_FACE = Path('/content/drive/MyDrive/sentinelid/data/faces')

if DRIVE_FACE.exists() and len(list(DRIVE_FACE.glob('train/*'))) > 100:
    print('MS1M found on Drive — symlinking...')
    FACE_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if not FACE_ROOT.exists():
        os.symlink(str(DRIVE_FACE), str(FACE_ROOT))
    n_ids = len(list(FACE_ROOT.glob('train/*')))
    print(f'✓ {n_ids} identities ready')
else:
    FACE_ROOT.mkdir(parents=True, exist_ok=True)
    try:
        print('Downloading MS1M-ArcFace (~10 GB) — ~20 min...')
        gdown.download(
            'https://drive.google.com/uc?id=1SigwBE6mPDqEMtmLkzAqkPIj8FMLJAOW',
            '/content/ms1m.zip', quiet=False
        )
        with zipfile.ZipFile('/content/ms1m.zip') as z:
            z.extractall(str(FACE_ROOT))
        DRIVE_FACE.mkdir(parents=True, exist_ok=True)
        shutil.copytree(str(FACE_ROOT), str(DRIVE_FACE), dirs_exist_ok=True)
        print('✓ MS1M-ArcFace ready')
    except Exception as e:
        print(f'⚠ Download failed ({e}) — synthetic fallback will run')
        FACE_ROOT = None

print('FACE_ROOT =', FACE_ROOT)

In [ ]:
# ── M5: MIDV-500 (document intelligence) ──────────────────────────────────────
import gdown, zipfile, os, shutil
from pathlib import Path

DOC_ROOT = Path('/content/data/documents')
DRIVE_DOC = Path('/content/drive/MyDrive/sentinelid/data/documents')

if DRIVE_DOC.exists() and sum(1 for _ in DRIVE_DOC.rglob('*.jpg')) > 500:
    print('MIDV-500 found on Drive — symlinking...')
    DOC_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if not DOC_ROOT.exists():
        os.symlink(str(DRIVE_DOC), str(DOC_ROOT))
    print(f'✓ {sum(1 for _ in DOC_ROOT.rglob("*.jpg"))} images ready')
else:
    DOC_ROOT.mkdir(parents=True, exist_ok=True)
    try:
        print('Downloading MIDV-500 (~2 GB)...')
        gdown.download(
            'https://drive.google.com/uc?id=1_i3_uKpkAGZlCO3OB-jMDbmRsR5-Z7A2',
            '/content/midv500.zip', quiet=False
        )
        with zipfile.ZipFile('/content/midv500.zip') as z:
            z.extractall(str(DOC_ROOT))
        DRIVE_DOC.mkdir(parents=True, exist_ok=True)
        shutil.copytree(str(DOC_ROOT), str(DRIVE_DOC), dirs_exist_ok=True)
        print('✓ MIDV-500 ready')
    except Exception as e:
        print(f'⚠ Download failed ({e}) — synthetic fallback will run')
        DOC_ROOT = None

print('DOC_ROOT =', DOC_ROOT)

In [ ]:
# ── Synthetic fallback — auto-fills any missing dataset ───────────────────────
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path

def _make_liveness(root, n=500, size=224):
    rng = np.random.default_rng(42)
    for split in ('train','val'):
        k = n if split=='train' else n//5
        for label in ('live','spoof'):
            d = Path(root)/split/label; d.mkdir(parents=True, exist_ok=True)
            have = len(list(d.glob('*.jpg')))
            for i in range(k - have):
                arr = (rng.normal(128,40,(size,size,3)).clip(0,255).astype(np.uint8) if label=='live'
                       else np.tile(rng.integers(50,200,3),(size,size,1)).astype(np.uint8))
                Image.fromarray(arr).save(d/f's_{label}_{have+i:05d}.jpg')
    print(f'  Liveness: {sum(1 for _ in Path(root).rglob("*.jpg"))} images')

def _make_deepfake(root, n=500, size=224):
    rng = np.random.default_rng(7)
    for split in ('train','val'):
        k = n if split=='train' else n//5
        for label in ('real','fake'):
            d = Path(root)/split/label; d.mkdir(parents=True, exist_ok=True)
            have = len(list(d.glob('*.jpg')))
            for i in range(k - have):
                arr = (rng.integers(0,256,(size,size,3),dtype=np.uint8) if label=='real'
                       else np.array(Image.fromarray(rng.integers(0,256,(16,16,3),dtype=np.uint8)).resize((size,size),Image.BILINEAR)))
                Image.fromarray(arr).save(d/f's_{label}_{have+i:05d}.jpg')
    print(f'  Deepfake: done')

def _make_face(root, n_ids=200, n_per=10, size=112):
    rng = np.random.default_rng(99)
    for split in ('train','val'):
        n = n_ids if split=='train' else max(20,n_ids//5)
        for uid in range(n):
            d = Path(root)/split/f'id_{uid:05d}'; d.mkdir(parents=True, exist_ok=True)
            if len(list(d.glob('*.jpg'))) >= n_per: continue
            base = rng.integers(40,220,3)
            for j in range(n_per):
                arr = np.clip(np.tile(base,(size,size,1))+rng.integers(-30,30,(size,size,3)),0,255).astype(np.uint8)
                Image.fromarray(arr).save(d/f'{j:04d}.jpg')
    print(f'  Face: {n_ids} identities')

def _make_documents(root, n=300, size=224):
    rng = np.random.default_rng(13)
    for split in ('train','val'):
        k = n if split=='train' else n//5
        for label in ('genuine','forged'):
            d = Path(root)/split/label; d.mkdir(parents=True, exist_ok=True)
            have = len(list(d.glob('*.jpg')))
            for i in range(k - have):
                img = Image.new('RGB',(size,size),tuple(rng.integers(240,255,3).tolist()))
                draw = ImageDraw.Draw(img)
                for row in range(5,200,20):
                    w = rng.integers(80,200)
                    draw.rectangle([10,row,10+w,row+8],fill=(0,0,0) if label=='genuine' else tuple(rng.integers(100,180,3).tolist()))
                if label=='forged':
                    arr = np.clip(np.array(img)+rng.integers(0,30,(size,size,3)),0,255).astype(np.uint8)
                    img = Image.fromarray(arr)
                img.save(d/f's_{label}_{have+i:04d}.jpg')
    print(f'  Documents: done')

print('Checking datasets...')
if 'LIVENESS_ROOT' not in dir() or LIVENESS_ROOT is None or not any(Path('/content/data/liveness').rglob('*.jpg')):
    print('Generating synthetic liveness...')
    _make_liveness('/content/data/liveness')
    LIVENESS_ROOT = Path('/content/data/liveness')

if 'DEEPFAKE_ROOT' not in dir() or DEEPFAKE_ROOT is None or not any(Path('/content/data/deepfake').rglob('*.jpg')):
    print('Generating synthetic deepfake...')
    _make_deepfake('/content/data/deepfake')
    DEEPFAKE_ROOT = Path('/content/data/deepfake')

if 'FACE_ROOT' not in dir() or FACE_ROOT is None or not any(Path('/content/data/faces').rglob('*.jpg')):
    print('Generating synthetic face IDs...')
    _make_face('/content/data/faces')
    FACE_ROOT = Path('/content/data/faces')

if 'DOC_ROOT' not in dir() or DOC_ROOT is None or not any(Path('/content/data/documents').rglob('*.jpg')):
    print('Generating synthetic documents...')
    _make_documents('/content/data/documents')
    DOC_ROOT = Path('/content/data/documents')

print('\n✅ All datasets ready.')
print(f'  Liveness : {sum(1 for _ in Path("/content/data/liveness").rglob("*.jpg")):,} images')
print(f'  Deepfake : {sum(1 for _ in Path("/content/data/deepfake").rglob("*.jpg")):,} images')
print(f'  Faces    : {len(list(Path("/content/data/faces/train").iterdir())) if Path("/content/data/faces/train").exists() else 0} identities')
print(f'  Documents: {sum(1 for _ in Path("/content/data/documents").rglob("*.jpg")):,} images')

## M1 · Passive 3D Liveness

In [ ]:
import yaml
from pathlib import Path

n_liveness = sum(1 for _ in Path('/content/data/liveness').rglob('*.jpg'))
using_real = n_liveness > 5000
print(f'Liveness: {n_liveness:,} images ({"real CelebA-Spoof" if using_real else "synthetic"})')

cfg_liveness = {
    'project': {'name': 'sentinelid-liveness', 'device': 'cuda', 'mixed_precision': True, 'compile_model': False},
    'paths': {'checkpoint_dir': str(DRIVE_CKPT)},
    'model': {'backbone': 'resnet50'},
    'data': {'image_size': 224, 'datasets': [{'name': 'liveness_train', 'root': '/content/data/liveness'}]},
    'training': {
        'batch_size': 64 if using_real else 32,
        'epochs': 30, 'lr': 1e-3, 'weight_decay': 1e-4,
        'bce_weight': 1.0, 'depth_weight': 0.5, 'contrastive_weight': 0.1,
        'contrastive_margin': 1.0, 'eval_every_n_epochs': 5
    },
    'compute': {'num_workers': 4, 'pin_memory': True, 'persistent_workers': True}
}
with open('/content/SentinelID/configs/liveness_config.yaml', 'w') as f:
    yaml.dump(cfg_liveness, f)
print('M1 config written.')

In [ ]:
!cd /content/SentinelID && python training/train_liveness.py --config configs/liveness_config.yaml

## M2 · Deepfake Detection

In [ ]:
n_df = sum(1 for _ in Path('/content/data/deepfake').rglob('*.jpg'))
using_real_df = n_df > 5000
print(f'Deepfake: {n_df:,} frames ({"real Celeb-DF v2" if using_real_df else "synthetic"})')

cfg_deepfake = {
    'project': {'name': 'sentinelid-deepfake', 'device': 'cuda', 'mixed_precision': True, 'compile_model': False},
    'paths': {'checkpoint_dir': str(DRIVE_CKPT)},
    'model': {'backbone': 'efficientnet_b4', 'pretrained': True},
    'data': {'image_size': 224, 'datasets': [{'name': 'deepfake_train', 'root': '/content/data/deepfake'}]},
    'training': {
        'batch_size': 64 if using_real_df else 32,
        'epochs': 20, 'lr': 5e-4, 'weight_decay': 1e-4,
        'focal_gamma': 2.0, 'eval_every_n_epochs': 5
    },
    'compute': {'num_workers': 4, 'pin_memory': True, 'persistent_workers': True}
}
with open('/content/SentinelID/configs/deepfake_config.yaml', 'w') as f:
    yaml.dump(cfg_deepfake, f)
print('M2 config written.')

In [ ]:
!cd /content/SentinelID && python training/train_deepfake.py --config configs/deepfake_config.yaml

## M3 · ArcFace Face Recognition

In [ ]:
from pathlib import Path

train_dir = Path('/content/data/faces/train')
n_ids = len([d for d in train_dir.iterdir() if d.is_dir()]) if train_dir.exists() else 0
using_real_face = n_ids > 1000
print(f'Face: {n_ids} identities ({"real MS1M" if using_real_face else "synthetic"})')

cfg_face = {
    'project': {'name': 'sentinelid-face', 'device': 'cuda', 'mixed_precision': True, 'compile_model': False},
    'paths': {'checkpoint_dir': str(DRIVE_CKPT)},
    'model': {
        'backbone': 'iresnet100' if using_real_face else 'iresnet50',
        'embedding_dim': 512,
        'num_classes': n_ids if n_ids > 0 else 200,
        'pretrained': False
    },
    'data': {'image_size': 112, 'datasets': [{'name': 'faces', 'root': '/content/data/faces'}]},
    'training': {
        'batch_size': 128 if using_real_face else 64,
        'epochs': 30, 'lr': 0.1 if using_real_face else 1e-3,
        'weight_decay': 5e-4, 's': 64.0, 'm': 0.5, 'eval_every_n_epochs': 5
    },
    'compute': {'num_workers': 4, 'pin_memory': True, 'persistent_workers': True}
}
with open('/content/SentinelID/configs/arcface_config.yaml', 'w') as f:
    yaml.dump(cfg_face, f)
print('M3 config written.')

In [ ]:
!cd /content/SentinelID && python training/train_face_recognition.py --config configs/arcface_config.yaml

## M4 · Behavioral Analysis (AU-GNN)

In [ ]:
import numpy as np
from pathlib import Path

beh_root = Path('/content/data/behavioral')
beh_root.mkdir(parents=True, exist_ok=True)

if not (beh_root / 'landmarks.npy').exists():
    rng = np.random.default_rng(42)
    n = 5000
    landmarks  = rng.uniform(0, 1, (n, 68, 2)).astype(np.float32)
    au_labels  = rng.uniform(0, 5, (n, 12)).astype(np.float32)
    gaze_labels = rng.uniform(-0.5, 0.5, (n, 2)).astype(np.float32)
    np.save(beh_root / 'landmarks.npy', landmarks)
    np.save(beh_root / 'au_labels.npy', au_labels)
    np.save(beh_root / 'gaze_labels.npy', gaze_labels)
    print(f'Behavioral data: {n} samples')
else:
    print(f'Behavioral data: {np.load(beh_root / "landmarks.npy").shape[0]} samples already present')

In [ ]:
cfg_behavioral = {
    'project': {'name': 'sentinelid-behavioral', 'device': 'cuda', 'mixed_precision': True, 'compile_model': False},
    'paths': {'checkpoint_dir': str(DRIVE_CKPT)},
    'model': {'hidden_dim': 256, 'n_layers': 3, 'n_heads': 4},
    'data': {'root': '/content/data/behavioral', 'val_split': 0.15},
    'training': {'batch_size': 256, 'epochs': 50, 'lr': 1e-3, 'weight_decay': 1e-4, 'au_weight': 1.0, 'gaze_weight': 0.5},
    'compute': {'num_workers': 4, 'pin_memory': True, 'persistent_workers': True}
}
with open('/content/SentinelID/configs/behavioral_config.yaml', 'w') as f:
    yaml.dump(cfg_behavioral, f)
!cd /content/SentinelID && python training/train_behavioral.py --config configs/behavioral_config.yaml

## M5 · Document Intelligence

In [ ]:
n_docs = sum(1 for _ in Path('/content/data/documents').rglob('*.jpg'))
using_real_doc = n_docs > 1000
print(f'Documents: {n_docs:,} images ({"real MIDV-500" if using_real_doc else "synthetic"})')

cfg_document = {
    'project': {'name': 'sentinelid-document', 'device': 'cuda', 'mixed_precision': True, 'compile_model': False},
    'paths': {'checkpoint_dir': str(DRIVE_CKPT)},
    'model': {'backbone': 'resnet50', 'num_classes': 2},
    'data': {'image_size': 224, 'root': '/content/data/documents'},
    'training': {'batch_size': 48 if using_real_doc else 32, 'epochs': 25, 'lr': 5e-4, 'weight_decay': 1e-4},
    'compute': {'num_workers': 4, 'pin_memory': True, 'persistent_workers': True}
}
with open('/content/SentinelID/configs/document_config.yaml', 'w') as f:
    yaml.dump(cfg_document, f)
!cd /content/SentinelID && python training/train_document.py --config configs/document_config.yaml

## M6 · Score Fusion

In [ ]:
import numpy as np
from pathlib import Path

score_root = Path('/content/data/score_vectors')
score_root.mkdir(parents=True, exist_ok=True)

if not (score_root / 'scores.npy').exists():
    rng = np.random.default_rng(0)
    n = 10000
    labels = rng.integers(0, 2, n)
    scores = np.zeros((n, 5), dtype=np.float32)
    for i, lbl in enumerate(labels):
        base = rng.beta(8,2,5) if lbl==1 else rng.beta(2,8,5)
        scores[i] = np.clip(base + rng.normal(0,0.03,5), 0, 1)
    np.save(score_root / 'scores.npy', scores)
    np.save(score_root / 'labels.npy', labels)
    print(f'Fusion data: {n} samples (accept={labels.sum()}, reject={n-labels.sum()})')

cfg_fusion = {
    'project': {'name': 'sentinelid-fusion', 'device': 'cuda', 'mixed_precision': False},
    'paths': {'checkpoint_dir': str(DRIVE_CKPT), 'score_cache_dir': '/content/data/score_vectors'},
    'model': {'hidden_dim': 256, 'n_modules': 5},
    'training': {'batch_size': 512, 'epochs': 300, 'lr': 1e-3, 'weight_decay': 1e-5, 'confidence_penalty': 0.1},
    'compute': {'num_workers': 0, 'pin_memory': False, 'persistent_workers': False}
}
with open('/content/SentinelID/configs/fusion_config.yaml', 'w') as f:
    yaml.dump(cfg_fusion, f)
!cd /content/SentinelID && python training/train_fusion.py --config configs/fusion_config.yaml

## M7 · Edge Distillation → ONNX

In [ ]:
from pathlib import Path

n_ids_distill = n_ids if n_ids > 0 else 200

cfg_distill = {
    'project': {'name': 'sentinelid-distill', 'device': 'cuda', 'mixed_precision': True, 'compile_model': False},
    'paths': {'checkpoint_dir': str(DRIVE_CKPT)},
    'face': {'num_classes': n_ids_distill},
    'student': {'face_embed_dim': 256, 'au_out': 1},
    'data': {'image_size': 224, 'datasets': [{'name': 'liveness_train', 'root': '/content/data/liveness'}]},
    'training': {
        'batch_size': 128, 'epochs': 40, 'lr': 1e-3, 'weight_decay': 1e-4,
        'temperature': 4.0, 'alpha': 0.7, 'embedding_weight': 0.3
    },
    'compute': {'num_workers': 4, 'pin_memory': True, 'persistent_workers': True}
}
with open('/content/SentinelID/configs/distillation_config.yaml', 'w') as f:
    yaml.dump(cfg_distill, f)
!pip install -q onnxscript
!cd /content/SentinelID && python training/distill_edge.py --config configs/distillation_config.yaml

## ✅ Verify All Checkpoints

In [ ]:
from pathlib import Path

ckpt_dir = Path('/content/drive/MyDrive/sentinelid/checkpoints')
expected = [
    ('liveness_best.pt',   'M1 Liveness'),
    ('deepfake_best.pt',   'M2 Deepfake'),
    ('face_model.pt',      'M3 Face Recognition'),
    ('behavioral_best.pt', 'M4 Behavioral'),
    ('document_best.pt',   'M5 Document'),
    ('fusion_best.pt',     'M6 Fusion'),
    ('edge_model.pt',      'M7 Edge (PyTorch)'),
    ('edge_model.onnx',    'M7 Edge (ONNX)'),
]

print('─'*65)
print(f'{"Module":<28} {"File":<24} {"Size":>8}  Status')
print('─'*65)
all_ok = True
for fname, label in expected:
    p = ckpt_dir / fname
    if p.exists():
        print(f'{label:<28} {fname:<24} {p.stat().st_size/1e6:>7.1f}M  ✓')
    else:
        print(f'{label:<28} {fname:<24} {"MISSING":>8}  ✗')
        all_ok = False
print('─'*65)
print('✅ All checkpoints present!' if all_ok else '⚠️  Some missing — re-run relevant module cells.')

## 🔍 ONNX Sanity Check + Latency Benchmark

In [ ]:
import onnxruntime as ort
import numpy as np, time
from pathlib import Path

onnx_path = str(Path('/content/drive/MyDrive/sentinelid/checkpoints/edge_model.onnx'))
sess = ort.InferenceSession(onnx_path, providers=['CUDAExecutionProvider','CPUExecutionProvider'])

print('Inputs: ', [(i.name, i.shape) for i in sess.get_inputs()])
print('Outputs:', [(o.name, o.shape) for o in sess.get_outputs()])

dummy4 = np.random.randn(4,3,224,224).astype(np.float32)
outs = sess.run(None, {'image': dummy4})
print(f'\nBatch=4: liveness={outs[0].shape} mean={outs[0].mean():.3f} | embed_norm={np.linalg.norm(outs[1],axis=1).mean():.3f}')

# Latency (batch=1, 100 runs)
dummy1 = np.random.randn(1,3,224,224).astype(np.float32)
for _ in range(10): sess.run(None, {'image': dummy1})  # warmup
times = []
for _ in range(100):
    t0 = time.perf_counter()
    sess.run(None, {'image': dummy1})
    times.append((time.perf_counter()-t0)*1000)
times = np.array(times)
print(f'\nLatency (batch=1, 100 runs):')
print(f'  Mean={times.mean():.1f}ms  P95={np.percentile(times,95):.1f}ms  P99={np.percentile(times,99):.1f}ms')
print(f'  Target <200ms mean: {"✅ PASS" if times.mean()<200 else "⚠️ FAIL"}')
print('\n✓ ONNX model verified.')

## 📊 Full ISO 30107-3 Evaluation

In [ ]:
!cd /content/SentinelID && python evaluation/evaluate_all.py \
    --checkpoints /content/drive/MyDrive/sentinelid/checkpoints \
    --data_root /content/data \
    --output /content/drive/MyDrive/sentinelid/results

## 📤 Push to GitHub

In [ ]:
import subprocess

repo = '/content/SentinelID'
subprocess.run(['git','config','user.email','aprameya.bharadwaj.05@gmail.com'], cwd=repo)
subprocess.run(['git','config','user.name','Aprameya Bharadwaj'], cwd=repo)

for cmd in [
    ['git','add','configs/','models/','training/','inference/','evaluation/','api/'],
    ['git','commit','-m','feat: all 7 modules trained; ONNX export verified; ISO eval suite added'],
    ['git','push'],
]:
    r = subprocess.run(cmd, cwd=repo, capture_output=True, text=True)
    print(r.stdout, r.stderr)